# 06. Limpieza y control de calidad final

Este notebook cierra la etapa premodelo del proyecto dejando listas las bases finales anual y mensual con reglas de limpieza explicitadas y controles de calidad reproducibles.

## Objetivos
- cargar la base anual de modelado y la base mensual integrada
- revisar duplicados, nulos, coberturas temporales y consistencia estructural
- distinguir entre faltantes esperados y faltantes problematicos
- aplicar ajustes finales de limpieza sin alterar la logica metodologica ya definida
- exportar dos salidas finales listas para EDA y siguientes fases

## Salidas esperadas
1. una base anual final limpia para modelado
2. una base mensual operativa limpia para analisis y dashboard
3. un resumen claro de validaciones superadas y nulos conservados por diseno


## Comentario metodologico

En esta etapa no se redefine el problema analitico ni se reconstruyen targets desde cero. La idea es cerrar el pipeline ya construido y dejar una capa de calidad final que permita avanzar al EDA y a la fase de modelos sin ambiguedades.

La logica general del notebook es la siguiente:

- la base anual se valida como capa principal de modelado a nivel `departamento-anio`
- la base mensual se valida como capa operativa y de soporte
- los faltantes esperados se conservan cuando representan ausencia real de observacion y no error de captura
- los flags y campos de apoyo se estandarizan para que las siguientes fases trabajen con criterios consistentes


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 160)


def find_project_root(start: Path) -> Path:
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / 'BASE_DE_DATOS').exists():
            return candidate
    raise FileNotFoundError('No se encontro una carpeta BASE_DE_DATOS en la ruta actual ni en sus padres.')


CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CURRENT_DIR)
BASE_DATOS = PROJECT_ROOT / 'BASE_DE_DATOS'

MONTHLY_PATH = BASE_DATOS / 'INTERMEDIAS' / 'base_mensual_integrada.csv'
ANNUAL_PATH = BASE_DATOS / 'FINALES' / 'dataset_modelado_anual.csv'
OUTPUT_DIR = BASE_DATOS / 'FINALES'
OUTPUT_ANNUAL = OUTPUT_DIR / 'dataset_modelado_anual_limpio.csv'
OUTPUT_MONTHLY = OUTPUT_DIR / 'dataset_operativo_mensual_limpio.csv'


In [2]:
for path in [MONTHLY_PATH, ANNUAL_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'No existe el archivo esperado: {path}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Rutas validadas correctamente.')
print('Proyecto:', PROJECT_ROOT)


Rutas validadas correctamente.
Proyecto: C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO


## Carga de datos

Se cargan las dos bases que alimentan esta etapa de cierre:

- base mensual integrada: soporte operativo y puente entre clima, satelite, precios y produccion
- base anual de modelado: capa principal para la siguiente fase analitica


In [3]:
monthly = pd.read_csv(MONTHLY_PATH, sep=';')
annual = pd.read_csv(ANNUAL_PATH, sep=';')

monthly['fecha'] = pd.to_datetime(monthly['fecha'], errors='coerce')
monthly = monthly.sort_values(['departamento', 'fecha']).reset_index(drop=True)
annual = annual.sort_values(['departamento', 'anio']).reset_index(drop=True)

print('Shape base mensual integrada:', monthly.shape)
print('Shape base anual modelado:', annual.shape)
print('Departamentos en mensual:', sorted(monthly['departamento'].dropna().unique().tolist()))
print('Departamentos en anual:', sorted(annual['departamento'].dropna().unique().tolist()))
print('Cobertura anual:', int(annual['anio'].min()), '-', int(annual['anio'].max()))
print('Cobertura mensual:', monthly['fecha'].min().strftime('%Y-%m'), '-', monthly['fecha'].max().strftime('%Y-%m'))

display(annual.head(3))
display(monthly.head(3))


Shape base mensual integrada: (628, 56)
Shape base anual modelado: (36, 76)
Departamentos en mensual: ['Cundinamarca', 'Risaralda']
Departamentos en anual: ['Cundinamarca', 'Risaralda']
Cobertura anual: 2007 - 2024
Cobertura mensual: 2000-01 - 2026-02


,departamento,anio,rendimiento_t_ha,produccion_t,area_cosechada_ha,area_sembrada_ha,rendimiento_referencia_fullsample,rendimiento_referencia_prev,rendimiento_referencia_prev3,perdida_rendimiento_anual_pct,perdida_rendimiento_anual_pct_prev,perdida_rendimiento_anual_pct_prev3,evento_perdida_anual,perdida_real_pct,evento_perdida_real,diff_target_vs_perdida_real_pct,precio_ico_usd_ton,precio_productor_usd_ton,correccion_aplicada,motivo_correccion,n_municipios,produccion_t_original,rendimiento_t_ha_original,delta_produccion_t,delta_rendimiento_t_ha,rendimiento_medio_municipal_reportado,dif_rendimiento_calculado_vs_reportado,rendimiento_medio_t_ha,produccion_media_t,precipitation_annual_sum,temp_aire_C_annual_mean,humedad_relativa_pct_annual_mean,soil_annual_mean,def_annual_mean,pet_annual_mean,aet_annual_mean,GDD_cafe_annual_mean,NDVI_annual_mean,EVI_annual_mean,LST_Day_1km_annual_mean,LST_Night_1km_annual_mean,Gpp_annual_mean,Lai_500m_annual_mean,Fpar_500m_annual_mean,NDVI_anomalia_pct_annual_mean,EVI_anomalia_pct_annual_mean,Gpp_anomalia_pct_annual_mean,Lai_500m_anomalia_pct_annual_mean,precipitation_anomalia_pct_annual_mean,indice_perdida_annual_mean,precipitation_cosecha_sum,temp_aire_C_cosecha_mean,humedad_relativa_pct_cosecha_mean,soil_cosecha_mean,def_cosecha_mean,pet_cosecha_mean,aet_cosecha_mean,GDD_cafe_cosecha_mean,NDVI_cosecha_mean,EVI_cosecha_mean,LST_Day_1km_cosecha_mean,LST_Night_1km_cosecha_mean,Gpp_cosecha_mean,Lai_500m_cosecha_mean,Fpar_500m_cosecha_mean,NDVI_anomalia_pct_cosecha_mean,EVI_anomalia_pct_cosecha_mean,Gpp_anomalia_pct_cosecha_mean,Lai_500m_anomalia_pct_cosecha_mean,precipitation_anomalia_pct_cosecha_mean,indice_perdida_cosecha_mean,elevacion_media_m,elevacion_std_m,pendiente_media,pendiente_std,aspecto_medio
0,Cundinamarca,2007,0.784083,33729.143730,43017.30,48195.69,0.925118,NaN,NaN,-15.245048,NaN,NaN,1,-15.245048,1.0,8.881784e-15,2750.0,2145.0,0.0,NaN,65.0,33729.143730,0.784083,0.000000,0.000000,0.629565,0.154518,0.925118,30017.744086,1687.793891,17.049639,78.108544,1041.084581,84.362900,914.525073,830.046049,7.766690,0.559274,0.358158,23.882376,11.317330,0.047687,1.857250,0.443968,-0.195160,-1.670301,-0.785513,2.800224,-6.275240,-0.883658,1098.347069,16.752371,82.927817,1187.465469,0.673783,836.745929,835.960358,7.622968,0.553130,0.358165,22.570806,11.000249,0.045257,1.793670,0.428554,-1.836137,-2.961987,-2.989640,0.026393,3.079408,-2.595921,1895.307361,1086.421734,14.454071,10.320921,185.295099
1,Cundinamarca,2008,0.818923,35732.355721,43633.35,48989.09,0.925118,0.784083,0.784083,-11.479062,4.443381,4.443381,0,-11.479062,0.0,1.243450e-14,2900.0,2262.0,1.0,Outlier de rendimiento de Cundinamarca 2008 corregido con promedio 2007-2009,66.0,78254.745626,1.793462,-42522.389905,-0.974539,1.908602,-0.115141,0.925118,30017.744086,1906.311791,16.676312,80.937549,1116.492910,24.495232,879.463229,854.818542,7.530276,0.549232,0.359143,23.041187,10.537202,0.046001,1.687877,0.420868,-2.102305,-1.553013,-4.110201,-7.360825,9.353536,-2.588506,1158.837926,16.641577,83.542475,1189.154816,0.285094,828.733359,828.321539,7.464702,0.547475,0.353955,22.929064,9.986722,0.044084,1.728891,0.426625,-3.050697,-4.335153,-5.295259,-5.319884,7.398212,-4.227036,1895.307361,1086.421734,14.454071,10.320921,185.295099
2,Cundinamarca,2009,0.853763,37118.057049,43475.84,48581.30,0.925118,0.801503,0.801503,-7.713077,6.520213,6.520213,0,-7.713077,0.0,1.065814e-14,2600.0,2028.0,0.0,NaN,69.0,37118.057049,0.853763,0.000000,0.000000,0.751595,0.102168,0.925118,30017.744086,1588.434373,17.222820,78.467313,1124.390811,29.993428,939.235693,909.166882,8.070870,0.554271,0.361746,23.927916,12.172007,0.048055,1.796530,0.435888,-0.992592,-0.657007,0.398230,-0.680473,-4.401427,-0.417123,888.102849,17.276536,79.594043,1163.311099,16.596303,919.117149,902.484833,8.187371,0.569247,0.371836,23.970807,11.922794,0.047742,1.940012,0.463285,0.868999,0.629689,2.547245,6.786979,-16.970570,1.348644,1895.307361,1086.421734,14.454071,10.320921,185.295099


,fecha,departamento,mes,anio,precipitation,temp_aire_C,humedad_relativa_pct,soil,def,pet,aet,GDD_cafe,NDVI,EVI,LST_Day_1km,LST_Night_1km,Gpp,Lai_500m,Fpar_500m,elevacion_media_m,elevacion_std_m,pendiente_media,pendiente_std,aspecto_medio,NDVI_anomalia_pct,EVI_anomalia_pct,Gpp_anomalia_pct,Lai_500m_anomalia_pct,precipitation_anomalia_pct,indice_perdida,evento_perdida,n_municipios,area_cosechada_ha,area_sembrada_ha,produccion_t_original,rendimiento_t_ha_original,produccion_t,rendimiento_t_ha,correccion_aplicada,motivo_correccion,delta_produccion_t,delta_rendimiento_t_ha,rendimiento_medio_municipal_reportado,dif_rendimiento_calculado_vs_reportado,rendimiento_medio_t_ha,produccion_media_t,factor_mensual_raw,factor_mensual,es_mes_cosecha,produccion_mensual_t,area_mensual_ha,perdida_real_pct,perdida_real_mensual_pct,evento_perdida_real,precio_ico_usd_ton,precio_productor_usd_ton
0,2000-01-01,Cundinamarca,1,2000,58.122070,16.040529,80.992858,1030.010331,87.170530,969.042186,881.660989,7.203958,NaN,NaN,NaN,NaN,0.045104,NaN,NaN,1895.307361,1086.421734,14.454071,10.320921,185.295099,NaN,NaN,-15.291502,NaN,-6.983227,-15.291502,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-02-01,Cundinamarca,2,2000,102.480240,16.364601,77.921106,1096.139029,4.391722,1015.409971,1010.790421,7.418556,0.275704,0.194953,25.120398,7.237869,0.046012,1.003104,0.286976,1895.307361,1086.421734,14.454071,10.320921,185.295099,-50.827307,-44.414822,-2.640640,-43.957202,25.536483,-32.627589,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-03-01,Cundinamarca,3,2000,126.669177,16.723544,77.873874,1071.270787,24.697898,1135.673349,1110.807486,7.621710,0.504174,0.331648,23.907992,11.393505,0.044515,1.484609,0.373975,1895.307361,1086.421734,14.454071,10.320921,185.295099,-3.017617,-2.038318,2.444149,-7.367417,-10.835857,-0.870595,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Reglas de control de calidad

Antes de limpiar, dejamos explicitas las reglas que vamos a usar para interpretar los datos.

- No se imputa el target anual principal.
- No se convierten faltantes de perdida real en ceros, porque ausencia de observacion no equivale a ausencia de evento.
- Los faltantes fuera del periodo observado de Agronet se consideran esperados en la base mensual.
- Las referencias historicas `prev` y `prev3` pueden quedar faltantes en el primer anio observado de cada departamento.
- Las columnas de apoyo y flags si se pueden estandarizar cuando ello mejora la trazabilidad del dataset.


In [4]:
ANNUAL_KEY_COLS = [
    'departamento', 'anio', 'rendimiento_t_ha', 'produccion_t', 'area_cosechada_ha',
    'area_sembrada_ha', 'perdida_rendimiento_anual_pct', 'evento_perdida_anual',
    'perdida_real_pct', 'precio_ico_usd_ton', 'precio_productor_usd_ton'
]

MONTHLY_PRODUCTIVE_COLS = [
    'n_municipios', 'area_cosechada_ha', 'area_sembrada_ha', 'produccion_t',
    'rendimiento_t_ha', 'factor_mensual', 'es_mes_cosecha', 'perdida_real_pct',
    'precio_ico_usd_ton', 'precio_productor_usd_ton'
]

MONTHLY_CLIMATE_COLS = [
    'precipitation', 'temp_aire_C', 'humedad_relativa_pct', 'soil', 'def', 'pet',
    'aet', 'GDD_cafe', 'NDVI', 'EVI', 'LST_Day_1km', 'LST_Night_1km', 'Gpp',
    'Lai_500m', 'Fpar_500m'
]


def summarize_nulls(df: pd.DataFrame, top_n: int = 15) -> pd.Series:
    return df.isna().sum().sort_values(ascending=False).head(top_n)


def max_abs_diff(a: pd.Series, b: pd.Series) -> float:
    aligned = pd.concat([a, b], axis=1)
    return float((aligned.iloc[:, 0] - aligned.iloc[:, 1]).abs().max())


## QC de la base anual

La base anual debe funcionar como dataset principal para el modelado. Por eso aqui priorizamos:

- unicidad por `departamento-anio`
- ausencia de faltantes en variables estructurales y target principal
- consistencia entre el target construido y la perdida anual ya venida del flujo anterior
- interpretacion correcta de los faltantes de referencias historicas auxiliares


In [5]:
annual_dup = int(annual.duplicated(['departamento', 'anio']).sum())
annual_key_nulls = annual[ANNUAL_KEY_COLS].isna().sum().sum()
annual_first_year_mask = annual.groupby('departamento')['anio'].transform('min') == annual['anio']

assert annual_dup == 0, 'La base anual tiene duplicados por departamento-anio.'
assert annual_key_nulls == 0, 'Hay faltantes en columnas clave de la base anual.'
assert annual['anio'].between(2007, 2024).all(), 'La cobertura anual se salio del rango esperado.'
assert set(annual['evento_perdida_anual'].dropna().unique()).issubset({0, 1}), 'evento_perdida_anual contiene valores fuera de 0/1.'
assert annual.loc[~annual_first_year_mask, 'rendimiento_referencia_prev'].notna().all(), 'Existen faltantes inesperados en la referencia prev.'
assert annual.loc[~annual_first_year_mask, 'perdida_rendimiento_anual_pct_prev'].notna().all(), 'Existen faltantes inesperados en la perdida prev.'
assert annual['diff_target_vs_perdida_real_pct'].abs().max() < 1e-10, 'El target anual principal no coincide con la perdida anual esperada.'

annual_qc_summary = pd.DataFrame([
    {'control': 'filas', 'resultado': int(len(annual))},
    {'control': 'columnas', 'resultado': int(annual.shape[1])},
    {'control': 'duplicados_departamento_anio', 'resultado': annual_dup},
    {'control': 'nulos_columnas_clave', 'resultado': int(annual_key_nulls)},
    {'control': 'max_abs_diff_target_vs_perdida_real_pct', 'resultado': float(annual['diff_target_vs_perdida_real_pct'].abs().max())},
    {'control': 'eventos_perdida_anual', 'resultado': int(annual['evento_perdida_anual'].sum())},
    {'control': 'correcciones_aplicadas', 'resultado': int(annual['correccion_aplicada'].fillna(0).sum())}
])

display(annual_qc_summary)
display(summarize_nulls(annual, top_n=12))


,control,resultado
0,filas,3.600000e+01
1,columnas,7.600000e+01
2,duplicados_departamento_anio,0.000000e+00
3,nulos_columnas_clave,0.000000e+00
4,max_abs_diff_target_vs_perdida_real_pct,1.953993e-14
5,eventos_perdida_anual,7.000000e+00
6,correcciones_aplicadas,1.000000e+00


motivo_correccion                      35
rendimiento_referencia_prev3            2
perdida_rendimiento_anual_pct_prev3     2
perdida_rendimiento_anual_pct_prev      2
rendimiento_referencia_prev             2
produccion_t                            0
anio                                    0
rendimiento_t_ha                        0
rendimiento_referencia_fullsample       0
area_sembrada_ha                        0
area_cosechada_ha                       0
departamento                            0
dtype: int64

## Limpieza final de la base anual

La limpieza final de la capa anual no elimina filas. Se limita a estandarizar flags y dejar mejor explicitos algunos metadatos utiles para la siguiente fase.


In [6]:
annual_clean = annual.copy()

annual_clean['correccion_aplicada'] = annual_clean['correccion_aplicada'].fillna(0).astype(int)
annual_clean['evento_perdida_real'] = annual_clean['evento_perdida_real'].fillna(annual_clean['evento_perdida_anual']).astype(int)
annual_clean['motivo_correccion'] = annual_clean['motivo_correccion'].fillna('sin_correccion')

annual_clean['disponible_referencia_prev'] = annual_clean['rendimiento_referencia_prev'].notna().astype(int)
annual_clean['disponible_referencia_prev3'] = annual_clean['rendimiento_referencia_prev3'].notna().astype(int)
annual_clean['listo_para_modelado_principal'] = annual_clean['perdida_rendimiento_anual_pct'].notna().astype(int)
annual_clean['version_target_principal'] = 'perdida_vs_referencia_fullsample'

display(annual_clean[['departamento', 'anio', 'correccion_aplicada', 'disponible_referencia_prev', 'disponible_referencia_prev3', 'listo_para_modelado_principal']].head(8))


,departamento,anio,correccion_aplicada,disponible_referencia_prev,disponible_referencia_prev3,listo_para_modelado_principal
0,Cundinamarca,2007,0,0,0,1
1,Cundinamarca,2008,1,1,1,1
2,Cundinamarca,2009,0,1,1,1
3,Cundinamarca,2010,0,1,1,1
4,Cundinamarca,2011,0,1,1,1
5,Cundinamarca,2012,0,1,1,1
6,Cundinamarca,2013,0,1,1,1
7,Cundinamarca,2014,0,1,1,1


## QC de la base mensual

La base mensual cumple un rol distinto: sirve como capa operativa, puente analitico y soporte para visualizacion. Por eso aqui separamos tres preguntas:

- si el panel mensual esta estructuralmente bien
- si la informacion productiva observada aparece donde debe aparecer
- si los faltantes de clima y satelite fuera del periodo observado responden a cola de serie y no a huecos arbitrarios


In [7]:
monthly_dup = int(monthly.duplicated(['fecha', 'departamento']).sum())
observed_mask = monthly['anio'].between(2007, 2024)
outside_observed_mask = ~observed_mask
months_per_year = monthly.groupby(['departamento', 'anio'])['mes'].nunique().rename('n_meses').reset_index()
complete_years = months_per_year['n_meses'].eq(12).sum()
incomplete_years = months_per_year['n_meses'].lt(12).sum()

assert monthly_dup == 0, 'La base mensual tiene duplicados por fecha-departamento.'
assert monthly['fecha'].notna().all(), 'Hay fechas invalidas en la base mensual.'
assert monthly['mes'].between(1, 12).all(), 'Existe al menos un mes fuera del rango 1-12.'
assert (monthly['fecha'].dt.month == monthly['mes']).all(), 'Existe inconsistencia entre fecha y mes.'
assert (monthly['fecha'].dt.year == monthly['anio']).all(), 'Existe inconsistencia entre fecha y anio.'
assert monthly.loc[observed_mask, MONTHLY_PRODUCTIVE_COLS].isna().sum().sum() == 0, 'Hay faltantes inesperados en columnas productivas dentro del periodo observado.'
assert monthly.loc[observed_mask, MONTHLY_CLIMATE_COLS].isna().sum().sum() == 0, 'Hay faltantes climaticos inesperados dentro del periodo observado.'
assert monthly.loc[outside_observed_mask, MONTHLY_PRODUCTIVE_COLS].notna().sum().sum() == 0, 'Aparecio informacion productiva fuera del periodo observado cuando no se esperaba.'

monthly_qc_summary = pd.DataFrame([
    {'control': 'filas', 'resultado': int(len(monthly))},
    {'control': 'columnas', 'resultado': int(monthly.shape[1])},
    {'control': 'duplicados_fecha_departamento', 'resultado': monthly_dup},
    {'control': 'anios_completos_con_12_meses', 'resultado': int(complete_years)},
    {'control': 'anios_incompletos', 'resultado': int(incomplete_years)},
    {'control': 'filas_periodo_agronet_observado', 'resultado': int(observed_mask.sum())},
    {'control': 'filas_fuera_periodo_agronet', 'resultado': int(outside_observed_mask.sum())}
])

display(monthly_qc_summary)
display(months_per_year.head(10))
display(summarize_nulls(monthly, top_n=18))


,control,resultado
0,filas,628
1,columnas,56
2,duplicados_fecha_departamento,0
3,anios_completos_con_12_meses,52
4,anios_incompletos,2
5,filas_periodo_agronet_observado,432
6,filas_fuera_periodo_agronet,196


,departamento,anio,n_meses
0,Cundinamarca,2000,12
1,Cundinamarca,2001,12
2,Cundinamarca,2002,12
3,Cundinamarca,2003,12
4,Cundinamarca,2004,12
5,Cundinamarca,2005,12
6,Cundinamarca,2006,12
7,Cundinamarca,2007,12
8,Cundinamarca,2008,12
9,Cundinamarca,2009,12


motivo_correccion                         616
perdida_real_mensual_pct                  412
dif_rendimiento_calculado_vs_reportado    196
precio_ico_usd_ton                        196
produccion_mensual_t                      196
area_mensual_ha                           196
evento_perdida_real                       196
perdida_real_pct                          196
rendimiento_t_ha_original                 196
delta_produccion_t                        196
rendimiento_medio_municipal_reportado     196
delta_rendimiento_t_ha                    196
rendimiento_medio_t_ha                    196
produccion_media_t                        196
factor_mensual_raw                        196
correccion_aplicada                       196
rendimiento_t_ha                          196
area_cosechada_ha                         196
dtype: int64

## Limpieza final de la base mensual

En la base mensual si hacemos una mejora operativa importante: propagamos el patron mensual de cosecha y ponderacion por `departamento-mes` a todos los anios del panel. Esto permite que la base operativa sea coherente incluso en anios sin observacion productiva de Agronet.

Aun asi, no imputamos produccion ni perdida real fuera del periodo observado.


In [8]:
monthly_clean = monthly.copy()

pattern_lookup = (
    monthly.loc[observed_mask, ['departamento', 'mes', 'factor_mensual_raw', 'factor_mensual', 'es_mes_cosecha']]
    .drop_duplicates()
    .sort_values(['departamento', 'mes'])
    .reset_index(drop=True)
)

assert pattern_lookup.groupby('departamento')['mes'].nunique().eq(12).all(), 'El patron mensual no cubre los 12 meses por departamento.'

pattern_lookup = pattern_lookup.rename(columns={
    'factor_mensual_raw': 'factor_mensual_raw_template',
    'factor_mensual': 'factor_mensual_template',
    'es_mes_cosecha': 'es_mes_cosecha_template'
})

monthly_clean = monthly_clean.merge(pattern_lookup, on=['departamento', 'mes'], how='left')

monthly_clean['factor_mensual_raw'] = monthly_clean['factor_mensual_raw'].fillna(monthly_clean['factor_mensual_raw_template'])
monthly_clean['factor_mensual'] = monthly_clean['factor_mensual'].fillna(monthly_clean['factor_mensual_template'])
monthly_clean['es_mes_cosecha'] = monthly_clean['es_mes_cosecha'].fillna(monthly_clean['es_mes_cosecha_template'])

monthly_clean = monthly_clean.drop(columns=['factor_mensual_raw_template', 'factor_mensual_template', 'es_mes_cosecha_template'])

monthly_clean['tiene_observacion_agronet'] = observed_mask.astype(int)
monthly_clean['evento_perdida_real_disponible'] = monthly_clean['perdida_real_pct'].notna().astype(int)
monthly_clean['correccion_aplicada'] = monthly_clean['correccion_aplicada'].fillna(0).astype(int)
monthly_clean['motivo_correccion'] = monthly_clean['motivo_correccion'].fillna('sin_correccion')
monthly_clean['es_mes_cosecha'] = monthly_clean['es_mes_cosecha'].astype(int)
monthly_clean['anio_completo'] = monthly_clean.groupby(['departamento', 'anio'])['mes'].transform('nunique').eq(12).astype(int)
monthly_clean['cobertura_climatica_completa'] = monthly_clean[MONTHLY_CLIMATE_COLS].notna().all(axis=1).astype(int)

display(monthly_clean[['departamento', 'fecha', 'factor_mensual', 'es_mes_cosecha', 'tiene_observacion_agronet', 'evento_perdida_real_disponible', 'anio_completo', 'cobertura_climatica_completa']].head(15))


,departamento,fecha,factor_mensual,es_mes_cosecha,tiene_observacion_agronet,evento_perdida_real_disponible,anio_completo,cobertura_climatica_completa
0,Cundinamarca,2000-01-01,0.042105,0,0,0,1,0
1,Cundinamarca,2000-02-01,0.036842,0,0,0,1,1
2,Cundinamarca,2000-03-01,0.047368,0,0,0,1,1
3,Cundinamarca,2000-04-01,0.078947,1,0,0,1,1
4,Cundinamarca,2000-05-01,0.094737,1,0,0,1,1
5,Cundinamarca,2000-06-01,0.084211,1,0,0,1,1
6,Cundinamarca,2000-07-01,0.057895,0,0,0,1,1
7,Cundinamarca,2000-08-01,0.052632,0,0,0,1,1
8,Cundinamarca,2000-09-01,0.063158,0,0,0,1,1
9,Cundinamarca,2000-10-01,0.126316,1,0,0,1,1


## Validaciones de consistencia entre capas

Una vez limpia cada base por separado, validamos que la informacion anual repetida dentro de la base mensual siga siendo consistente con la base anual final.


In [9]:
factor_sum_complete = (
    monthly_clean.loc[monthly_clean['anio_completo'].eq(1)]
    .groupby(['departamento', 'anio'])['factor_mensual']
    .sum()
)

assert np.isclose(factor_sum_complete, 1.0).all(), 'La suma de factor_mensual no es 1.0 en los anios completos.'

monthly_bridge = (
    monthly_clean.loc[monthly_clean['tiene_observacion_agronet'].eq(1), ['departamento', 'anio', 'produccion_t', 'rendimiento_t_ha', 'area_cosechada_ha', 'area_sembrada_ha', 'precio_ico_usd_ton', 'precio_productor_usd_ton']]
    .groupby(['departamento', 'anio'], as_index=False)
    .first()
)

consistency = annual_clean.merge(monthly_bridge, on=['departamento', 'anio'], suffixes=('_annual', '_monthly'))
compare_cols = ['produccion_t', 'rendimiento_t_ha', 'area_cosechada_ha', 'area_sembrada_ha', 'precio_ico_usd_ton', 'precio_productor_usd_ton']

consistency_rows = []
for col in compare_cols:
    diff = (consistency[f'{col}_annual'] - consistency[f'{col}_monthly']).abs()
    max_diff = float(diff.max())
    assert max_diff < 1e-10, f'La columna {col} no coincide entre capas anual y mensual.'
    consistency_rows.append({'columna': col, 'max_abs_diff': max_diff})

consistency_report = pd.DataFrame(consistency_rows)
display(consistency_report)

retained_nulls = pd.DataFrame([
    {'caso': 'referencias_prev_anual', 'nulos': int(annual_clean['rendimiento_referencia_prev'].isna().sum()), 'lectura': 'esperado en el primer anio observado por departamento'},
    {'caso': 'referencias_prev3_anual', 'nulos': int(annual_clean['rendimiento_referencia_prev3'].isna().sum()), 'lectura': 'esperado en el primer anio observado por departamento'},
    {'caso': 'perdida_real_mensual_pct', 'nulos': int(monthly_clean['perdida_real_mensual_pct'].isna().sum()), 'lectura': 'esperado fuera de meses con proxy mensual y fuera de Agronet'},
    {'caso': 'clima_fuera_periodo_observado', 'nulos': int(monthly_clean.loc[outside_observed_mask, MONTHLY_CLIMATE_COLS].isna().sum().sum()), 'lectura': 'esperado en la cola final de algunas series satelitales y climaticas'}
])
display(retained_nulls)


,columna,max_abs_diff
0,produccion_t,0.0
1,rendimiento_t_ha,0.0
2,area_cosechada_ha,0.0
3,area_sembrada_ha,0.0
4,precio_ico_usd_ton,0.0
5,precio_productor_usd_ton,0.0


,caso,nulos,lectura
0,referencias_prev_anual,2,esperado en el primer anio observado por departamento
1,referencias_prev3_anual,2,esperado en el primer anio observado por departamento
2,perdida_real_mensual_pct,412,esperado fuera de meses con proxy mensual y fuera de Agronet
3,clima_fuera_periodo_observado,184,esperado en la cola final de algunas series satelitales y climaticas


## Exportacion de salidas finales

Se exportan dos archivos nuevos para no sobrescribir las bases fuente de esta etapa:

- `dataset_modelado_anual_limpio.csv`
- `dataset_operativo_mensual_limpio.csv`


In [10]:
annual_clean = annual_clean.sort_values(['departamento', 'anio']).reset_index(drop=True)
monthly_clean = monthly_clean.sort_values(['departamento', 'fecha']).reset_index(drop=True)

annual_clean.to_csv(OUTPUT_ANNUAL, sep=';', index=False, encoding='utf-8')
monthly_clean.to_csv(OUTPUT_MONTHLY, sep=';', index=False, encoding='utf-8')

print('Archivo anual exportado en:', OUTPUT_ANNUAL)
print('Archivo mensual exportado en:', OUTPUT_MONTHLY)


Archivo anual exportado en: C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO\BASE_DE_DATOS\FINALES\dataset_modelado_anual_limpio.csv
Archivo mensual exportado en: C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO\BASE_DE_DATOS\FINALES\dataset_operativo_mensual_limpio.csv


In [11]:
final_summary = pd.DataFrame([
    {'salida': 'dataset_modelado_anual_limpio.csv', 'filas': int(annual_clean.shape[0]), 'columnas': int(annual_clean.shape[1]), 'comentario': 'base final para EDA anual y modelado'},
    {'salida': 'dataset_operativo_mensual_limpio.csv', 'filas': int(monthly_clean.shape[0]), 'columnas': int(monthly_clean.shape[1]), 'comentario': 'base final operativa para analisis mensual y dashboard'}
])

display(final_summary)
print('Notebook 06 completado correctamente.')


,salida,filas,columnas,comentario
0,dataset_modelado_anual_limpio.csv,36,80,base final para EDA anual y modelado
1,dataset_operativo_mensual_limpio.csv,628,60,base final operativa para analisis mensual y dashboard


Notebook 06 completado correctamente.
